In [16]:
import pandas as pd

In [17]:
results = pd.read_csv("results.csv")
results = results[results["tournament"] == "FIFA World Cup"].copy()
results = results[results["date"].str.split("-").str[0].astype(int) >= 2002].copy()

team_tournament_features = pd.read_csv("team_tournament_features.csv")

In [18]:
len(results)

488

In [19]:
results["home_team"] = results["home_team"].str.strip()
results["away_team"] = results["away_team"].str.strip()
results["home_score"] = results["home_score"].astype("Int64")
results["away_score"] = results["away_score"].astype("Int64")

results["home_win"] = results["home_score"] > results["away_score"]

In [20]:
results['neutral'].value_counts()

neutral
True     439
False     49
Name: count, dtype: int64

In [21]:
results[results['neutral'] == False]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_win
26448,2002-06-04,Japan,Belgium,2,2,FIFA World Cup,Saitama,Japan,False,False
26449,2002-06-04,South Korea,Poland,2,0,FIFA World Cup,Busan,South Korea,False,True
26466,2002-06-09,Japan,Russia,1,0,FIFA World Cup,Yokohama,Japan,False,True
26469,2002-06-10,South Korea,United States,1,1,FIFA World Cup,Daegu,South Korea,False,False
26485,2002-06-14,Japan,Tunisia,2,0,FIFA World Cup,Osaka,Japan,False,True
26486,2002-06-14,South Korea,Portugal,1,0,FIFA World Cup,Incheon,South Korea,False,True
26494,2002-06-18,Japan,Turkey,0,1,FIFA World Cup,Rifu,Japan,False,False
26495,2002-06-18,South Korea,Italy,2,1,FIFA World Cup,Daejeon,South Korea,False,True
26499,2002-06-22,South Korea,Spain,0,0,FIFA World Cup,Gwangju,South Korea,False,False
26502,2002-06-25,South Korea,Germany,0,1,FIFA World Cup,Seoul,South Korea,False,False


In [35]:
len(set(team_tournament_features['team_name'].unique()))

71

In [36]:
unique_teams = pd.unique(results[['home_team', 'away_team']].values.ravel())
print(len(set(unique_teams)))

71


In [24]:
missing_teams = set(team_tournament_features['team_name'].unique()) - set(unique_teams)
missing_teams

{'Serbia and Montenegro'}

In [25]:
team_tournament_features[team_tournament_features['team_name'].isin(missing_teams)]

,tournament_id,team_id,team_name,year,top5_league_count,connectivity,population,gdp_usd,gdp_per_capita,fifa_rank
54,WC-2006,T-67,Serbia and Montenegro,2006,4,0,8055030,2.994040e+10,3716.98,44


In [26]:
results[(results["home_team"] == "Serbia") | (results["away_team"] == "Serbia") | (results["home_team"] == "Montenegro") | (results["away_team"] == "Montenegro")]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_win
30047,2006-06-11,Serbia,Netherlands,0,1,FIFA World Cup,Leipzig,Germany,True,False
30060,2006-06-16,Argentina,Serbia,6,0,FIFA World Cup,Gelsenkirchen,Germany,True,True
30077,2006-06-21,Ivory Coast,Serbia,3,2,FIFA World Cup,Munich,Germany,True,True
33874,2010-06-13,Serbia,Ghana,0,1,FIFA World Cup,Pretoria,South Africa,True,False
33888,2010-06-18,Germany,Serbia,0,1,FIFA World Cup,Port Elizabeth,South Africa,True,False
33909,2010-06-23,Australia,Serbia,2,1,FIFA World Cup,Nelspruit,South Africa,True,True
41650,2018-06-17,Costa Rica,Serbia,0,1,FIFA World Cup,Samara,Russia,True,False
41667,2018-06-22,Serbia,Switzerland,1,2,FIFA World Cup,Kaliningrad,Russia,True,False
41684,2018-06-27,Serbia,Brazil,0,2,FIFA World Cup,Moscow,Russia,True,False
45733,2022-11-24,Brazil,Serbia,2,0,FIFA World Cup,Lusail,Qatar,True,True


In [27]:
team_tournament_features['team_name'] = team_tournament_features['team_name'].replace(
    'Serbia and Montenegro', 'Serbia'
)

In [28]:
results["year"] = results["date"].str.split("-").str[0].astype(int)

home_features = team_tournament_features.rename(columns={
    "team_name": "home_team",
    "top5_league_count": "home_top5_league_count",
    "connectivity": "home_connectivity",
})[["home_team", "year", "home_top5_league_count", "home_connectivity"]]

away_features = team_tournament_features.rename(columns={
    "team_name": "away_team",
    "top5_league_count": "away_top5_league_count",
    "connectivity": "away_connectivity",
})[["away_team", "year", "away_top5_league_count", "away_connectivity"]]

results = results.merge(home_features, on=["home_team", "year"], how="left")
results = results.merge(away_features, on=["away_team", "year"], how="left")

results["top5_league_count_diff"] = results["home_top5_league_count"] - results["away_top5_league_count"]
results["connectivity_diff"] = results["home_connectivity"] - results["away_connectivity"]

In [29]:
unmatched = results[results["home_top5_league_count"].isna() | results["away_top5_league_count"].isna()]
print(f"unmatched rows after merge: {len(unmatched)}")
unmatched[["date", "home_team", "away_team", "year"]]

unmatched rows after merge: 0


,date,home_team,away_team,year


In [30]:
results

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_win,year,home_top5_league_count,home_connectivity,away_top5_league_count,away_connectivity,top5_league_count_diff,connectivity_diff
0,2002-05-31,France,Senegal,0,1,FIFA World Cup,Seoul,South Korea,True,False,2002,23,10,17,12,6,-2
1,2002-06-01,Germany,Saudi Arabia,8,0,FIFA World Cup,Sapporo,Japan,True,True,2002,23,26,0,0,23,26
2,2002-06-01,Republic of Ireland,Cameroon,1,1,FIFA World Cup,Niigata,Japan,True,False,2002,17,9,13,1,4,8
3,2002-06-01,Uruguay,Denmark,1,2,FIFA World Cup,Ulsan,South Korea,True,False,2002,13,4,9,1,4,3
4,2002-06-02,Argentina,Nigeria,1,0,FIFA World Cup,Kashima,Japan,True,True,2002,18,8,6,1,12,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,2026-07-11,Argentina,Switzerland,3,1,FIFA World Cup,Kansas City,United States,True,True,2026,20,16,23,1,-3,15
484,2026-07-14,France,Spain,0,2,FIFA World Cup,Arlington,United States,True,False,2026,24,14,26,37,-2,-23
485,2026-07-15,England,Argentina,1,2,FIFA World Cup,Atlanta,United States,True,False,2026,25,17,20,16,5,1
486,2026-07-18,France,England,4,6,FIFA World Cup,Miami Gardens,United States,True,False,2026,24,14,25,17,-1,-3


In [31]:
country_cols = ["population", "gdp_usd", "gdp_per_capita", "fifa_rank"]

home_country_features = team_tournament_features.rename(
    columns={"team_name": "home_team", **{c: f"home_{c}" for c in country_cols}}
)[["home_team", "year"] + [f"home_{c}" for c in country_cols]]

away_country_features = team_tournament_features.rename(
    columns={"team_name": "away_team", **{c: f"away_{c}" for c in country_cols}}
)[["away_team", "year"] + [f"away_{c}" for c in country_cols]]

results = results.merge(home_country_features, on=["home_team", "year"], how="left")
results = results.merge(away_country_features, on=["away_team", "year"], how="left")

for c in country_cols:
    results[f"{c}_diff"] = results[f"home_{c}"] - results[f"away_{c}"]

In [32]:
unmatched_country = results[results["home_fifa_rank"].isna() | results["away_fifa_rank"].isna()]
print(f"unmatched rows after country-feature merge: {len(unmatched_country)}")
unmatched_country[["date", "home_team", "away_team", "year"]]

unmatched rows after country-feature merge: 0


,date,home_team,away_team,year


In [33]:
results

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_win,...,home_gdp_per_capita,home_fifa_rank,away_population,away_gdp_usd,away_gdp_per_capita,away_fifa_rank,population_diff,gdp_usd_diff,gdp_per_capita_diff,fifa_rank_diff
0,2002-05-31,France,Senegal,0,1,FIFA World Cup,Seoul,South Korea,True,False,...,22450.44,1,10212942,6.507825e+09,637.21,42,51151435,1.371150e+12,21813.23,-41
1,2002-06-01,Germany,Saudi Arabia,8,0,FIFA World Cup,Sapporo,Japan,True,True,...,23628.33,11,17041397,1.841376e+11,10805.31,34,65308528,1.761653e+12,12823.02,-23
2,2002-06-01,Republic of Ireland,Cameroon,1,1,FIFA World Cup,Niigata,Japan,True,False,...,28282.41,15,15309490,1.095349e+10,715.47,17,-11443247,9.839318e+10,27566.94,-2
3,2002-06-01,Uruguay,Denmark,1,2,FIFA World Cup,Ulsan,South Korea,True,False,...,6382.76,24,5358783,1.647914e+11,30751.65,20,-2084532,-1.438927e+11,-24368.89,4
4,2002-06-02,Argentina,Nigeria,1,0,FIFA World Cup,Kashima,Japan,True,True,...,7141.48,2,129862595,7.355784e+10,566.43,27,-92237770,1.951389e+11,6575.05,-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,2026-07-11,Argentina,Switzerland,3,1,FIFA World Cup,Kansas City,United States,True,True,...,14018.50,1,9005582,8.849404e+11,98265.76,19,36690577,-2.443490e+11,-84247.26,-18
484,2026-07-14,France,Spain,0,2,FIFA World Cup,Arlington,United States,True,False,...,44213.44,3,48848840,1.580695e+12,32358.90,2,19702813,1.450209e+12,11854.54,1
485,2026-07-15,England,Argentina,1,2,FIFA World Cup,Atlanta,United States,True,False,...,49221.21,4,45696159,6.405914e+11,14018.50,1,12661359,2.231836e+12,35202.71,3
486,2026-07-18,France,England,4,6,FIFA World Cup,Miami Gardens,United States,True,False,...,44213.44,3,58357518,2.872428e+12,49221.21,4,10194135,1.584762e+11,-5007.77,-1


In [34]:
results.iloc[results["home_team"] == "Serbia"]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_win,...,home_gdp_per_capita,home_fifa_rank,away_population,away_gdp_usd,away_gdp_per_capita,away_fifa_rank,population_diff,gdp_usd_diff,gdp_per_capita_diff,fifa_rank_diff
71,2006-06-11,Serbia,Netherlands,0,1,FIFA World Cup,Leipzig,Germany,True,False,...,3716.98,44,16319868,6.853482e+11,41994.71,3,-8264838,-6.554078e+11,-38277.73,41
135,2010-06-13,Serbia,Ghana,0,1,FIFA World Cup,Pretoria,South Africa,True,False,...,6169.11,15,24862664,2.604872e+10,1047.70,32,-17541857,1.911417e+10,5121.41,-17
281,2018-06-22,Serbia,Switzerland,1,2,FIFA World Cup,Kaliningrad,Russia,True,False,...,6292.55,34,8451840,6.952008e+11,82254.38,6,-1430982,-6.510218e+11,-75961.83,28
298,2018-06-27,Serbia,Brazil,0,2,FIFA World Cup,Moscow,Russia,True,False,...,6292.55,34,204703445,2.063515e+12,10080.51,2,-197682587,-2.019336e+12,-3787.96,32
364,2022-12-02,Serbia,Switzerland,2,3,FIFA World Cup,Doha,Qatar,True,False,...,9232.96,21,8704546,8.134088e+11,93446.43,15,-1870220,-7.503077e+11,-84213.47,6
